<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔧 Lab 03a: Add Function Tools to Your Travel Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 What We Learn

In this lab, we give the Contoso Travel agent the ability to **look up real data** using **Function Tools**. We define Python functions that search our flight, hotel, and car rental CSV data, then register them as tools the agent can call. This transforms the concierge from a generic chatbot into a data-driven travel assistant.

> **Think of tools as superpowers.** Without them, the agent can only talk. With them, it can look up flights, search hotel inventory, and check car availability — turning a chatbot into a data-driven assistant.

The tool-call loop works like this:

```
User Query → Agent (LLM decides to call a tool)
                → function_call request
                    → Python function executes
                        → results returned to Agent
                            → Agent synthesizes final answer
```

---


## 🧳 The Contoso Travel Story

The Contoso Travel Concierge from Lab 02 can hold a great conversation — it's warm, knowledgeable about travel, and handles follow-up questions naturally. But there's a critical problem that became obvious during the first round of internal testing. A tester asked: *"Find me a flight from Seattle to Paris under $800."* The concierge confidently recommended a flight... **that doesn't exist.** It hallucinated an airline, a price, and a departure time. Another tester asked about hotels in Tokyo and got a recommendation for a property that was never in Contoso's inventory.

### 🔴 The Problem You Face Right Now

- **Your agent is making things up.** Without access to real data, it fills in the gaps with plausible-sounding but fabricated information.
- For a travel agency, this is disastrous — customers book based on these recommendations.
- You need the agent to search Contoso's actual flight, hotel, and car rental inventory and return only verified results.
- But how do you connect an AI agent to structured data sources?

### ✅ What This Lab Solves

This lab introduces **Function Tools** — the mechanism that lets your agent call Python functions to query real data:
 - defining search functions for flights, hotels, and car rentals,
 - registering them as tools the agent can invoke, and
 - handling the call-and-response loop.

By the end, when a customer asks about flights to Paris, the agent searches the actual CSV inventory and returns only real options.

## 2. Setup

---


In [ ]:
# Setup: imports include FunctionTool for registering
# callable tools, and Tool as the base type list hint
import os
import json
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, Tool, FunctionTool

# .env lives one level above the notebooks dir
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry")

## 3. Load the Travel Data

Load the travel CSV files into pandas DataFrames. The tool functions query these DataFrames.

---


In [ ]:
# Path is relative to notebook location:
# notebooks/1-prompt-agents/ -> ../../data/\n
data_path = "../../data/contoso-travel"

flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")

print(f"✈️  Loaded {len(flights_df)} flights")
print(f"🏨 Loaded {len(hotels_df)} hotels")
print(f"🚗 Loaded {len(cars_df)} car rentals")

## 4. Define Tool Functions

Each function searches the travel data based on query parameters and returns matching results as JSON.

---


In [ ]:
# Tool functions must return JSON strings (not dicts).
# The agent receives the raw string as tool output.

def search_flights(origin: str = None, destination: str = None, cabin_class: str = None, max_price: float = None) -> str:
    """Search for available flights based on criteria."""
    results = flights_df.copy()  # avoid mutating source
    if origin:
        # Case-insensitive match for user flexibility
        results = results[results["origin"].str.lower() == origin.lower()]
    if destination:
        results = results[results["destination"].str.lower() == destination.lower()]
    if cabin_class:
        results = results[results["cabin_class"].str.lower() == cabin_class.lower()]
    if max_price:
        results = results[results["price_usd"] <= max_price]
    
    if results.empty:
        return json.dumps({"message": "No flights found matching your criteria.", "results": []})
    
    # orient="records" -> list of dicts, easiest for LLM
    return results.to_json(orient="records")


def search_hotels(city: str = None, min_stars: int = None, max_price: float = None, amenities: str = None) -> str:
    """Search for available hotels based on criteria."""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if min_stars:
        results = results[results["star_rating"] >= min_stars]
    if max_price:
        results = results[results["price_per_night_usd"] <= max_price]
    if amenities:
        # Substring match: "Pool" matches "Pool, Spa, WiFi"
        results = results[results["amenities"].str.lower().str.contains(amenities.lower())]
    
    if results.empty:
        return json.dumps({"message": "No hotels found matching your criteria.", "results": []})
    
    return results.to_json(orient="records")


def search_car_rentals(city: str = None, car_type: str = None, max_price: float = None) -> str:
    """Search for available car rentals based on criteria."""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if car_type:
        results = results[results["car_type"].str.lower() == car_type.lower()]
    if max_price:
        results = results[results["price_per_day_usd"] <= max_price]
    # Always filter to available-only (unlike flights/hotels)
    results = results[results["available"] == True]
    
    if results.empty:
        return json.dumps({"message": "No car rentals found matching your criteria.", "results": []})
    
    return results.to_json(orient="records")


# Quick test
print("🧪 Testing search_flights('Seattle', 'Paris'):")
print(search_flights(origin="Seattle", destination="Paris"))

## 5. Register Function Tools

We create `FunctionTool` definitions with JSON schemas that tell the agent what parameters each function accepts.

---


In [ ]:
# FunctionTool: JSON Schema tells the agent what params
# each function accepts. The agent generates arguments
# conforming to this schema when it decides to call a tool.

flight_tool = FunctionTool(
    name="search_flights",
    description="Search for available flights. Use this when the customer asks about flights, airfare, or flying to a destination.",
    parameters={
        "type": "object",
        "properties": {
            "origin": {"type": "string", "description": "Departure city name"},
            "destination": {"type": "string", "description": "Arrival city name"},
            # enum constrains values the model can generate
            "cabin_class": {"type": "string", "description": "Cabin class: Economy, Business, or First", "enum": ["Economy", "Business", "First"]},
            "max_price": {"type": "number", "description": "Maximum ticket price in USD"},
        },
        "required": [],  # all params optional for flexible queries
        "additionalProperties": False,
    },
    strict=False,  # allows optional params (strict requires all)
)

hotel_tool = FunctionTool(
    name="search_hotels",
    description="Search for available hotels. Use this when the customer asks about hotels, accommodation, or places to stay.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name to search hotels in"},
            "min_stars": {"type": "integer", "description": "Minimum star rating (1-5)"},
            "max_price": {"type": "number", "description": "Maximum price per night in USD"},
            "amenities": {"type": "string", "description": "Desired amenity to filter by (e.g., Pool, Spa, WiFi)"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

car_rental_tool = FunctionTool(
    name="search_car_rentals",
    description="Search for available car rentals. Use this when the customer asks about rental cars or ground transportation.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name for car pickup"},
            "car_type": {"type": "string", "description": "Type of car: Economy, SUV, Luxury, or Minivan", "enum": ["Economy", "SUV", "Luxury", "Minivan"]},
            "max_price": {"type": "number", "description": "Maximum price per day in USD"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

tools: list[Tool] = [flight_tool, hotel_tool, car_rental_tool]
print(f"✅ Defined {len(tools)} function tools: {[t.name for t in tools]}")

## 6. Create the Enhanced Travel Agent

Now we create the agent with the function tools attached.

---


In [ ]:
# Define the travel agent's system instructions
TRAVEL_AGENT_INSTRUCTIONS = """You are the Contoso Travel Agent, an expert travel assistant with access to real-time travel data.

You have access to the following tools:
- search_flights: Find available flights between cities
- search_hotels: Find hotels in a destination city  
- search_car_rentals: Find car rental options in a city

When a customer asks about travel options:
1. Use the appropriate tool(s) to search our inventory
2. Present the results in a clear, organized format
3. Include relevant details like prices, ratings, and availability
4. Offer helpful recommendations based on the results

Always be helpful, accurate, and professional. If no results match, suggest alternative criteria."""

# tools= attaches FunctionTool definitions so the agent
# knows which functions it can invoke during a response
agent = project_client.agents.create_version(
    agent_name="contoso-travel-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=TRAVEL_AGENT_INSTRUCTIONS,
        tools=tools,
    ),
)

print(f"✅ Travel Agent created with {len(tools)} tools!")
print(f"   Name: {agent.name}")
print(f"   Version: {agent.version}")

## 7. Test: Flight Search

Test the agent with a flight search query. The agent calls the `search_flights` function tool.

---


In [ ]:
# FunctionCallOutput: the type used to return tool
# results back to the model for synthesis
from openai.types.responses.response_input_param import FunctionCallOutput

def handle_tool_calls(response):
    """Process any function_call output items and return results."""
    # Map tool names to local Python functions
    tool_map = {
        "search_flights": search_flights,
        "search_hotels": search_hotels,
        "search_car_rentals": search_car_rentals,
    }
    
    function_results = []
    for item in response.output:
        # "function_call" items = agent requesting a tool
        if item.type == "function_call":
            func = tool_map.get(item.name)
            if func:
                # arguments is a JSON string, not a dict
                args = json.loads(item.arguments)
                print(f"   🔧 Calling {item.name}({args})")
                result = func(**args)
                # call_id links this result to the request
                function_results.append(
                    FunctionCallOutput(
                        type="function_call_output",
                        call_id=item.call_id,
                        output=result,
                    )
                )
    return function_results

# Start a new conversation and trigger a tool call
conversation = openai_client.conversations.create()

response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Find me flights from Seattle to Paris under $800.",
)

# Response contains function_call items, not text yet
print("📨 Agent requested tool calls:")
function_results = handle_tool_calls(response)

## 8. Handle Function Call Responses

The agent issued a function call. Execute the function and return the results so the agent can generate its final response.

---


In [ ]:
# Send tool results back so agent can generate a
# natural-language answer from the raw data.
# previous_response_id chains this to the prior turn.
if function_results:
    response = openai_client.responses.create(
        input=function_results,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

print(f"\n🤖 Travel Agent Response:\n")
print(response.output_text)

## 9. Test: Hotel + Car Combo

Test a more complex query that triggers multiple tool calls.

---


In [ ]:
# Multi-tool: agent may call tools sequentially across
# rounds (hotel first, then car) rather than all at once
conversation2 = openai_client.conversations.create()

response = openai_client.responses.create(
    conversation=conversation2.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="I need a 4-star hotel in Tokyo and a car rental there too. Budget is $200/night for hotel.",
)

print("📨 Agent requested tool calls:")
function_results = handle_tool_calls(response)

if function_results:
    # Return first batch of tool results
    response = openai_client.responses.create(
        input=function_results,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

    # Agent may issue a 2nd round of tool calls
    # (e.g., it searched hotels first, now wants cars)
    more_results = handle_tool_calls(response)
    if more_results:
        response = openai_client.responses.create(
            input=more_results,
            previous_response_id=response.id,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )

print(f"\n🤖 Travel Agent Response:\n")
print(response.output_text)

## 10. Cleanup

---


In [ ]:
# Cleanup: delete both conversations and agent version
openai_client.conversations.delete(conversation_id=conversation.id)
openai_client.conversations.delete(conversation_id=conversation2.id)
print("✅ Conversations deleted")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ Agent version deleted")

openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed")

## 11. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Defined Python functions that search CSV data for flights, hotels, and car rentals</li>
  <li>Created <code>FunctionTool</code> definitions with JSON schemas</li>
  <li>Built an enhanced travel agent with all three tools attached</li>
  <li>Handled the function call → execute → return results workflow</li>
  <li>Tested single-tool and multi-tool queries</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 Key Takeaway:</strong> Function tools transform the Contoso Travel agent from a general-purpose chatbot into a data-driven assistant that can query actual inventory. The agent intelligently decides which tools to call based on the customer's query.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ Next:</strong> In <a href="lab-03b-workflow.ipynb">Lab 03b</a>, we'll create a <b>multi-agent workflow</b> with specialized Flight, Hotel, and Car Rental agents orchestrated by a workflow definition!
</div>

---
